# M5.3 Transients and the Frequency Domain

Book pointer: chapter 10, section 10.2 (circuits with capacitors); chapter 16, sections 16.1 (circuit components), 16.2 (circuits in the time domain), 16.3 (circuits in the frequency domain).

## 1. Recall

Answer on paper before running any cell.

1. A capacitor $C$ charges through a resistor $R$ from a battery $V_0$. Write $V_C(t)$ and $I(t)$. What fraction of $V_0$ is reached after one time constant, and after five?
2. A series RLC circuit is switched onto a battery. Write the equation for the capacitor voltage, its characteristic equation, and the condition on $R$ that separates the underdamped, critically damped and overdamped regimes.
3. Write the complex impedances $Z_R$, $Z_C$ and $Z_L$ at angular frequency $\omega$. Which one leads the current by $90^\circ$ and which lags it?
4. Derive the transfer function $H(\omega) = V_{\text{out}}/V_{\text{in}}$ of an RC low-pass filter. What is $|H|$ at $\omega = 1/RC$, in dB, and what is the high-frequency slope in dB per decade?
5. Define the quality factor $Q$ of a series RLC circuit. How is the bandwidth between the half-power points related to $\omega_0$ and $Q$?

## 2. Principle to algorithm

A circuit with capacitors and inductors has memory: the charge on each capacitor and the current in each inductor cannot jump. Those quantities are the state of the circuit. Kirchhoff's laws, with $I = C\,dV_C/dt$ for a capacitor and $V_L = L\,dI_L/dt$ for an inductor, turn into one first-order ODE per capacitor voltage and one per inductor current. That is a system $d\mathbf{y}/dt = \mathbf{f}(t, \mathbf{y})$, and `integrators.rk4_ode` integrates it on a time grid. This is the time-domain route: it handles any source waveform, including switching, and it shows the transient.

When every source is a sinusoid at one angular frequency $\omega$ and the transient has died away, every voltage and current is a sinusoid at the same $\omega$, fixed by an amplitude and a phase. Writing each as the real part of a complex amplitude times $e^{j\omega t}$ turns $d/dt$ into multiplication by $j\omega$. A capacitor becomes the admittance $j\omega C$ and an inductor $1/(j\omega L)$, and the circuit becomes a resistor network with complex conductances. `circuits.mna` with `omega` given solves that network by the same nodal analysis used for DC in notebook M5.2. This is the frequency-domain route: one linear solve per frequency, no transient, no time step.

The two routes describe the same circuit, so they must agree: integrate the ODEs long enough for the transient to decay, fit a sinusoid to the last few periods, and compare its amplitude and phase with the phasor. That agreement is the main check of this notebook. The closed forms for RC charging, the three RLC regimes, the RC ladder and the resonance width test each route separately first.

Two sign conventions matter. A source phasor $V_0$ (real) means $v(t) = V_0\cos\omega t$. `mna` returns the current through each voltage source from its first node to its second, inside the source; a source that delivers current $I$ to the circuit through its first (positive) node reports $-I$.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import circuits as ci
from emlib.integrators import rk4_ode

plt.rcParams["figure.figsize"] = (6, 4)
np.set_printoptions(precision=4, suppress=True)


def steady_phasor(t, y, omega, n_periods):
    # Least-squares fit of y(t) = a cos(omega t) + b sin(omega t) over the last n_periods.
    # Re[(a - j b) e^{j omega t}] = a cos + b sin, so the phasor is a - j b.
    keep = t >= t[-1] - n_periods * 2 * np.pi / omega
    basis = np.stack([np.cos(omega * t[keep]), np.sin(omega * t[keep])], axis=1)
    (a, b), *_ = np.linalg.lstsq(basis, y[keep], rcond=None)
    return a - 1j * b

### RC charging and discharging

With the source at $V_0$ the loop equation is $V_0 = RI + V_C$ with $I = C\,dV_C/dt$, so
$$\frac{dV_C}{dt} = \frac{V_0 - V_C}{RC}, \qquad V_C(t) = V_0\left(1 - e^{-t/RC}\right).$$
After the source is switched to zero at $t_1$ the capacitor discharges, $V_C = V_C(t_1)\,e^{-(t - t_1)/RC}$. The time constant is $\tau = RC$: $63\%$ of the way after one $\tau$, $99.3\%$ after five.

Two checks. RK4 has a global error proportional to $\Delta t^4$, so halving the step should cut the error by 16. And the energy dissipated in the resistor during a full charge, $\int I^2R\,dt$, equals the energy $\tfrac12CV_0^2$ left in the capacitor, whatever $R$ is: charging a capacitor from a fixed voltage is never more than $50\%$ efficient.

In [ ]:
R, C, V0 = 1e3, 1e-6, 1.0
tau = R * C
charge = lambda t, v: (V0 - v) / tau
discharge = lambda t, v: -v / tau

t1 = np.linspace(0, 5 * tau, 1001)
v1 = rk4_ode(charge, 0.0, t1)
t2 = np.linspace(5 * tau, 10 * tau, 1001)
v2 = rk4_ode(discharge, v1[-1], t2)
print(f"charging:    max |V_C - closed form| = {np.abs(v1 - ci.rc_charging(t1, R, C, V0)).max():.1e} V")
print(f"discharging: max |V_C - closed form| = {np.abs(v2 - v1[-1] * np.exp(-(t2 - t2[0]) / tau)).max():.1e} V")
print(f"V_C(tau) / V0 = {np.interp(tau, t1, v1):.4f}   (1 - 1/e = {1 - np.exp(-1):.4f})")

print("\nsteps per tau   max error    ratio to the next")
errs = []
for n in (5, 10, 20, 40):
    t = np.linspace(0, 5 * tau, 5 * n + 1)
    errs.append(np.abs(rk4_ode(charge, 0.0, t) - ci.rc_charging(t, R, C, V0)).max())
for n, e, nxt in zip((5, 10, 20, 40), errs, errs[1:] + [np.nan]):
    print(f"{n:13d}   {e:10.2e}   {e / nxt:8.2f}")

t_long = np.linspace(0, 10 * tau, 4001)
v_long = rk4_ode(charge, 0.0, t_long)
heat = np.trapezoid(((V0 - v_long) / R) ** 2 * R, t_long)
stored = 0.5 * C * v_long[-1] ** 2
print(f"\nafter 10 tau: dissipated {heat * 1e6:.4f} uJ, stored {stored * 1e6:.4f} uJ, (1/2) C V0^2 = {0.5 * C * V0**2 * 1e6:.4f} uJ")

plt.plot(t1 / tau, v1, label="charging, RK4")
plt.plot(t2 / tau, v2, label="discharging, RK4")
plt.plot(t1 / tau, V0 - v1, "--", label="V_R = I R while charging")
plt.axvline(1, color="grey", ls=":"); plt.xlabel("t / RC"); plt.ylabel("V"); plt.legend(); plt.title("RC charge and discharge"); plt.show()

### Series RLC: the three regimes

A battery $V_0$ is switched at $t = 0$ onto $R$, $L$ and $C$ in series, all initially at rest. The state is $(V_C, I)$:
$$\frac{dV_C}{dt} = \frac{I}{C}, \qquad \frac{dI}{dt} = \frac{V_0 - RI - V_C}{L}.$$
Eliminating $I$ gives $LC\,\ddot V_C + RC\,\dot V_C + V_C = V_0$, with characteristic equation $s^2 + 2\alpha s + \omega_0^2 = 0$, where $\alpha = R/2L$ and $\omega_0 = 1/\sqrt{LC}$. The roots are $s = -\alpha \pm \sqrt{\alpha^2 - \omega_0^2}$, and the sign under the root sets the regime:

- underdamped, $R < 2\sqrt{L/C}$: $V_C = V_0\left[1 - e^{-\alpha t}\left(\cos\omega_d t + \frac{\alpha}{\omega_d}\sin\omega_d t\right)\right]$ with $\omega_d = \sqrt{\omega_0^2 - \alpha^2}$; the voltage overshoots and rings, with its first peak at $t = \pi/\omega_d$ reaching $V_0(1 + e^{-\alpha\pi/\omega_d})$, which is $1.605\,V_0$ for $R = 10\,\Omega$ below;
- critically damped, $R = 2\sqrt{L/C}$: $V_C = V_0\left[1 - e^{-\alpha t}(1 + \alpha t)\right]$, the fastest approach without overshoot;
- overdamped, $R > 2\sqrt{L/C}$: $V_C = V_0\left[1 + \dfrac{s_2e^{s_1t} - s_1e^{s_2t}}{s_1 - s_2}\right]$ with two real negative roots; the slower one dominates and the approach is sluggish.

With $L = 1$ mH and $C = 1\,\mu$F, $\omega_0 = 31\,623$ rad/s and the critical resistance is $2\sqrt{L/C} = 63.2\,\Omega$.

In [ ]:
L, C, V0 = 1e-3, 1e-6, 1.0
w0 = ci.resonance_omega(L, C)
R_crit = 2 * np.sqrt(L / C)
print(f"omega0 = {w0:.0f} rad/s (f0 = {w0 / 2 / np.pi:.0f} Hz), critical R = {R_crit:.2f} ohm")


def rlc_step_closed(t, R, L, C, V0):
    a, w0 = R / (2 * L), 1 / np.sqrt(L * C)
    if np.isclose(a, w0):
        return V0 * (1 - np.exp(-a * t) * (1 + a * t))
    if a < w0:
        wd = np.sqrt(w0**2 - a**2)
        return V0 * (1 - np.exp(-a * t) * (np.cos(wd * t) + a / wd * np.sin(wd * t)))
    s1, s2 = -a + np.sqrt(a**2 - w0**2), -a - np.sqrt(a**2 - w0**2)
    return V0 * (1 + (s2 * np.exp(s1 * t) - s1 * np.exp(s2 * t)) / (s1 - s2))


t = np.linspace(0, 2e-3, 4001)
for R_, name in ((10.0, "underdamped"), (R_crit, "critical"), (300.0, "overdamped")):
    rhs = lambda tk, y, R_=R_: np.array([y[1] / C, (V0 - R_ * y[1] - y[0]) / L])
    y = rk4_ode(rhs, np.zeros(2), t)
    err = np.abs(y[:, 0] - rlc_step_closed(t, R_, L, C, V0)).max()
    print(f"R = {R_:6.2f} ohm  {name:12s}: max |V_C - closed form| = {err:.1e} V, peak V_C = {y[:, 0].max():.3f} V")
    plt.plot(t * 1e3, y[:, 0], label=f"{name}, R = {R_:.1f}")
plt.axhline(V0, color="grey", ls=":"); plt.xlabel("t [ms]"); plt.ylabel("V_C [V]"); plt.legend(); plt.title("series RLC step response"); plt.show()

### Sinusoidal steady state: the ODE and the phasor

Now drive the series circuit with $v(t) = V_0\cos\omega t$ from rest. The ODE solution is the sum of a transient, which decays as $e^{-\alpha t}$ whatever $\omega$ is, and a steady sinusoid at the driving frequency. After about $20/\alpha$ the transient is below $10^{-8}$ of its starting size. A least-squares fit of $a\cos\omega t + b\sin\omega t$ to the last few periods then gives the steady current, whose phasor is $a - jb$ (the helper `steady_phasor`).

The frequency-domain answer is one call to `mna` with the source phasor $V_0$, a resistor, an inductor and a capacitor. The loop current is minus the reported source current (section 2). Below, $R = 10\,\Omega$ and $\omega = 0.7\,\omega_0$, below resonance, where the capacitor dominates and the current leads the voltage.

In [ ]:
R = 10.0


def series_rlc_ode_current(omega, R, L, C, V0=1.0, n_periods=6, steps_per_period=200):
    # Integrate the driven series RLC from rest and return (t, I(t), fitted phasor).
    alpha = R / (2 * L)
    T = 2 * np.pi / omega
    t_end = 20 / alpha + n_periods * T
    dt = min(T / steps_per_period, 0.02 / alpha)
    t = np.linspace(0, t_end, int(np.ceil(t_end / dt)) + 1)
    rhs = lambda tk, y: np.array([y[1] / C, (V0 * np.cos(omega * tk) - R * y[1] - y[0]) / L])
    I = rk4_ode(rhs, np.zeros(2), t)[:, 1]
    return t, I, steady_phasor(t, I, omega, n_periods)


def series_rlc_mna_current(omega, R, L, C, V0=1.0):
    _, I_vs = ci.mna(4, resistors=[(1, 2, R)], inductors=[(2, 3, L)], capacitors=[(3, 0, C)], voltage_sources=[(1, 0, V0)], omega=omega)
    return -I_vs[0]


w = 0.7 * w0
t, I_t, I_fit = series_rlc_ode_current(w, R, L, C)
I_ph = series_rlc_mna_current(w, R, L, C)
print(f"omega = 0.7 omega0: ODE fit |I| = {abs(I_fit) * 1e3:.5f} mA, phase {np.degrees(np.angle(I_fit)):+.4f} deg")
print(f"                    phasor  |I| = {abs(I_ph) * 1e3:.5f} mA, phase {np.degrees(np.angle(I_ph)):+.4f} deg")
print(f"relative amplitude difference {abs(abs(I_fit) / abs(I_ph) - 1):.1e}, phase difference {abs(np.angle(I_fit / I_ph)):.1e} rad")

plt.figure(figsize=(8, 3.5))
plt.plot(t * 1e3, I_t * 1e3, lw=0.8, label="RK4 from rest")
plt.plot(t * 1e3, np.real(I_ph * np.exp(1j * w * t)) * 1e3, "--", lw=0.8, label="phasor steady state")
plt.xlim(0, 2.5); plt.xlabel("t [ms]"); plt.ylabel("I [mA]"); plt.legend(); plt.title("driven series RLC: the transient dies, the phasor remains"); plt.show()

### A three-stage RC ladder: Bode plot and loading

One RC low-pass stage has $H = 1/(1 + x)$ with $x = j\omega RC$: flat below $\omega_c = 1/RC$, falling $20$ dB per decade above it. Three stages in a row should fall $60$ dB per decade. If each stage were buffered so that it did not load the one before, the response would be the cascade $1/(1 + x)^3$. A plain ladder of three identical sections is not buffered: each capacitor draws current through the resistors of the earlier stages. Chaining the three sections (series $R$, then shunt $C$) and leaving the output open gives
$$H_{\text{ladder}} = \frac{1}{1 + 6x + 5x^2 + x^3},$$
which you can check by writing Kirchhoff's current law at the three capacitor nodes, starting from the output and working back. The extra terms move the corner down: the $-3$ dB point of the cascade is at $\omega RC = \sqrt{2^{1/3} - 1} = 0.510$, and the ladder's is at $0.194$, more than a factor of two lower. The same polynomial, with $x = sRC$, gives the ladder's three natural time constants, $5.05$, $0.64$ and $0.31\,RC$: the slowest sets how long the ODE route must run before the transient is gone. The high-frequency slope is the same, because there only the $x^3$ term survives.

The plot shows `mna` on the ladder, the polynomial, and the cascade. The dots are the ODE route: three capacitor-voltage equations integrated to steady state and fitted.

In [ ]:
Rl, Cl = 1e3, 100e-9
wc = 1 / (Rl * Cl)
ladder = dict(resistors=[(1, 2, Rl), (2, 3, Rl), (3, 4, Rl)], capacitors=[(2, 0, Cl), (3, 0, Cl), (4, 0, Cl)], voltage_sources=[(1, 0, 1.0)])
H_mna = lambda omega: ci.mna(5, omega=omega, **ladder)[0][4]
H_poly = lambda omega: 1 / (1 + 6 * (1j * omega * Rl * Cl) + 5 * (1j * omega * Rl * Cl) ** 2 + (1j * omega * Rl * Cl) ** 3)
H_cascade = lambda omega: 1 / (1 + 1j * omega * Rl * Cl) ** 3

ws = wc * np.logspace(-2, 3, 301)
H = np.array([H_mna(wk) for wk in ws])
print(f"max |H_mna - H_polynomial| over the sweep = {np.abs(H - H_poly(ws)).max():.1e}")

roots = np.roots([1, 5, 6, 1])                      # x^3 + 5x^2 + 6x + 1 = 0, x = s RC
print("natural time constants of the ladder, in units of RC:", np.sort(-1 / roots.real)[::-1])

dB = 20 * np.log10(np.abs(H))
w3_ladder = np.interp(-3.0103, dB[::-1], ws[::-1]) / wc
print(f"-3 dB point: ladder omega RC = {w3_ladder:.3f}, buffered cascade {np.sqrt(2 ** (1 / 3) - 1):.3f}")


def ladder_ode_phasor(omega, n_periods=6):
    tau_slow = Rl * Cl * (-1 / roots.real).max()
    T = 2 * np.pi / omega
    t_end = 30 * tau_slow + n_periods * T
    dt = min(T / 200, Rl * Cl / 20)
    t = np.linspace(0, t_end, int(np.ceil(t_end / dt)) + 1)

    def rhs(tk, v):
        vs = np.cos(omega * tk)
        i1, i2, i3 = (vs - v[0]) / Rl, (v[0] - v[1]) / Rl, (v[1] - v[2]) / Rl
        return np.array([i1 - i2, i2 - i3, i3]) / Cl

    v = rk4_ode(rhs, np.zeros(3), t)
    return steady_phasor(t, v[:, 2], omega, n_periods)


w_ode = wc * np.array([0.3, 1.0, 3.0])
H_ode = np.array([ladder_ode_phasor(wk) for wk in w_ode])
for wk, ho in zip(w_ode, H_ode):
    hm = H_mna(wk)
    print(f"omega RC = {wk / wc:3.1f}: |H| ODE {abs(ho):.6f}  mna {abs(hm):.6f}   phase ODE {np.degrees(np.angle(ho)):+8.3f}  mna {np.degrees(np.angle(hm)):+8.3f} deg")

fig, ax = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
ax[0].semilogx(ws / wc, dB, label="ladder (mna)")
ax[0].semilogx(ws / wc, 20 * np.log10(np.abs(H_cascade(ws))), "--", label="buffered cascade")
ax[0].semilogx(w_ode / wc, 20 * np.log10(np.abs(H_ode)), "o", label="ladder, ODE steady state")
ax[0].set_ylabel("|H| [dB]"); ax[0].set_ylim(-150, 5); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].semilogx(ws / wc, np.degrees(np.unwrap(np.angle(H))), label="ladder")
ax[1].semilogx(ws / wc, np.degrees(np.unwrap(np.angle(H_cascade(ws)))), "--", label="cascade")
ax[1].semilogx(w_ode / wc, np.degrees(np.angle(H_ode)) - 360 * (np.angle(H_ode) > 0), "o")   # same branch as the unwrapped curve
ax[1].set_ylabel("phase [deg]"); ax[1].set_xlabel("omega RC"); ax[1].grid(alpha=0.3)
fig.suptitle("three-stage RC low-pass"); plt.show()

### Resonance and Q

For the driven series RLC the current phasor is $I = V_0/Z$ with $Z = R + j\left(\omega L - \dfrac{1}{\omega C}\right)$. The reactance vanishes at $\omega_0 = 1/\sqrt{LC}$: the inductor and capacitor voltages are equal and opposite and cancel, the circuit looks like $R$ alone, and the current peaks at $V_0/R$ in phase with the source. The quality factor $Q = \omega_0L/R = \sqrt{L/C}/R$ measures how sharp the peak is. The power $\tfrac12|I|^2R$ falls to half its peak where the reactance equals $\pm R$, which gives two frequencies $\omega_\pm$ with
$$\omega_+ - \omega_- = \frac{R}{L} = \frac{\omega_0}{Q}.$$
At resonance the voltage across the capacitor is $Q$ times the source voltage, which is how a high-$Q$ circuit picks out one frequency.

**Animation.** Phasors for $V_R$, $V_L$ and $V_C$ of a series RLC rotate at the driving frequency, drawn head to tail so that they add up to the source phasor, and their projections on the real axis trace out the actual voltages. Then the frequency sweeps through resonance: $V_L$ and $V_C$ grow and shrink in opposition and cancel exactly at $\omega_0$, where the current peaks. Watch the readout of $\omega/\omega_0$ and $|I|$.

In [ ]:
from emlib import show
show.animation("m5_circuits", "PhasorRLC")

In [ ]:
def half_power_width(r_axis, I_abs):
    # Width between the two crossings of |I|max / sqrt(2), in units of omega0, by linear interpolation.
    k = int(np.argmax(I_abs))
    half = I_abs.max() / np.sqrt(2)
    lo = np.interp(half, I_abs[:k + 1], r_axis[:k + 1])
    hi = np.interp(half, I_abs[k:][::-1], r_axis[k:][::-1])
    return lo, hi


r_axis = np.linspace(0.2, 3.0, 5601)
for R_ in (5.0, 10.0, 30.0):
    I_abs = np.abs([series_rlc_mna_current(r * w0, R_, L, C) for r in r_axis])
    Q = ci.quality_factor(R_, L, C)
    lo, hi = half_power_width(r_axis, I_abs)
    print(f"R = {R_:4.1f} ohm: Q = {Q:5.2f}, peak at omega/omega0 = {r_axis[np.argmax(I_abs)]:.4f}, peak |I| R / V0 = {I_abs.max() * R_:.6f}, "
          f"half-power points {lo:.4f} and {hi:.4f}, width {hi - lo:.4f} omega0 (1/Q = {1 / Q:.4f})")
    plt.plot(r_axis, I_abs * 1e3, label=f"R = {R_:.0f} ohm, Q = {Q:.1f}")
plt.xlim(0.2, 2.5); plt.xlabel("omega / omega0"); plt.ylabel("|I| [mA] for V0 = 1 V"); plt.legend(); plt.title("series RLC resonance"); plt.show()

### Exercise: the current phasor by hand

Replace the body of `series_rlc_current_phasor` with your own expression for the current phasor of a series RLC circuit driven by $V_0\cos\omega t$, from the impedances of the three components. The check compares it with `mna` over a range of frequencies and resistances.

In [ ]:
def series_rlc_current_phasor(omega, R, L, C, V0):
    # Return the complex current amplitude I such that i(t) = Re[I exp(j omega t)].
    return V0 / (R + 1j * omega * L + 1 / (1j * omega * C))           # replace with your own version


worst = 0.0
for R_ in (1.0, 10.0, 100.0):
    for r in (0.1, 0.5, 1.0, 2.0, 10.0):
        ref = series_rlc_mna_current(r * w0, R_, L, C, V0=2.0)
        mine = series_rlc_current_phasor(r * w0, R_, L, C, 2.0)
        worst = max(worst, abs(mine - ref) / abs(ref))
print(f"largest relative difference from mna: {worst:.1e}")
assert worst < 1e-10

## 4. Verify

Three checks with asserts.

1. Time domain against frequency domain: the driven series RLC at five frequencies from $0.3\,\omega_0$ to $3\,\omega_0$. The fitted amplitude must match the phasor to a relative error below $10^{-3}$, and the phase to $10^{-3}$ rad.
2. Resonance: with $R = 10\,\Omega$ the peak of $|I|$ from `mna` sits at `resonance_omega` and the half-power width, found by interpolating the two crossings of $|I|_{\max}/\sqrt2$ on a fine sweep, equals $\omega_0/Q = R/L$. The sweep must contain both crossings: for low $Q$ the upper one lies well above $\omega_0$ (at $1.58\,\omega_0$ for $R = 30\,\Omega$), because the curve is not symmetric in $\omega$.
3. The three-stage ladder falls at $-60$ dB per decade well above its corner: the slope between $100\,\omega_c$ and $1000\,\omega_c$.

In [ ]:
R = 10.0
print("omega/omega0    |I| ODE [mA]   |I| phasor [mA]   rel. amp. error   phase error [rad]")
ratios = np.array([0.3, 0.7, 1.0, 1.5, 3.0])
amp_err, ph_err, I_ode_all = [], [], []
for r in ratios:
    _, _, I_fit = series_rlc_ode_current(r * w0, R, L, C)
    I_ph = series_rlc_mna_current(r * w0, R, L, C)
    amp_err.append(abs(abs(I_fit) / abs(I_ph) - 1)); ph_err.append(abs(np.angle(I_fit / I_ph))); I_ode_all.append(abs(I_fit))
    print(f"{r:12.2f}   {abs(I_fit) * 1e3:12.6f}   {abs(I_ph) * 1e3:15.6f}   {amp_err[-1]:15.1e}   {ph_err[-1]:17.1e}")
assert max(amp_err) < 1e-3 and max(ph_err) < 1e-3

r_fine = np.linspace(0.25, 3.1, 11401)
I_abs = np.abs([series_rlc_mna_current(r * w0, R, L, C) for r in r_fine])
k = int(np.argmax(I_abs))
half = I_abs.max() / np.sqrt(2)
lo, hi = half_power_width(r_fine, I_abs)
width = (hi - lo) * w0
print(f"\npeak at omega/omega0 = {r_fine[k]:.5f}; half-power width = {width:.2f} rad/s, R/L = omega0/Q = {R / L:.2f} rad/s, "
      f"relative difference {abs(width / (R / L) - 1):.1e}")
assert abs(r_fine[k] - 1) < 1e-3 and abs(width / (R / L) - 1) < 1e-3

slope = 20 * np.log10(abs(H_mna(1000 * wc)) / abs(H_mna(100 * wc)))
print(f"ladder slope between 100 and 1000 omega_c: {slope:.3f} dB per decade")
assert abs(slope + 60) < 0.1

plt.plot(r_fine, I_abs * 1e3, label="phasor (mna)")
plt.plot(ratios, np.array(I_ode_all) * 1e3, "o", label="ODE steady state (RK4 + fit)")
plt.axhline(half * 1e3, color="grey", ls=":"); plt.axvline(lo, color="grey", ls=":"); plt.axvline(hi, color="grey", ls=":")
plt.xlim(0.25, 3.1); plt.xlabel("omega / omega0"); plt.ylabel("|I| [mA]"); plt.legend(); plt.title("two routes, one resonance curve"); plt.show()

## 5. Explore and rebuild

Explore. A square wave of amplitude $V_0$ and angular frequency $\omega$ is the sum $\frac{4V_0}{\pi}\sum_{k\ \text{odd}}\frac{1}{k}\sin k\omega t$. Each harmonic is a phasor problem on its own, and by linearity the response is the sum of the responses. Drive the series RLC with a square wave at $\omega_0/3$: build the steady-state current from the first twenty odd harmonics with `mna`, and compare it with RK4 driven by the square wave itself. Which harmonic dominates the current, and why? What happens to the agreement near the jumps of the square wave as you add harmonics? Then replace the series circuit by a band-pass made from a parallel LC tank in series with a resistor, and explain why the tank's impedance, not its current, peaks at $\omega_0$.

Rebuild. Close this notebook. In a blank editor, write the right-hand side of the series RLC ODE for an arbitrary source $v(t)$, the least-squares phasor fit, and the `mna` call for the same circuit, then reproduce the five-frequency table of section 4.